# Automated FinTech Portfolio Analysis Project

This project fetches stock market data using Python and prepares data for portfolio analysis and dashboard visualization.

!pip install yfinance  #yfinance is a finance data specific python library that will fetch data from yahoo finance site

In [1]:
import yfinance as yf 
import pandas as pd

In [2]:
stock_dictionary = {
    "Finance": ["JPM", "MS", "GS", "BAC", "C"],
    "Technology": ["AAPL", "MSFT", "AMZN", "GOOG", "NVDA"],
    "Healthcare": ["PFE", "JNJ", "MRK", "UNH", "ABBV"],
    "Automobile": ["TSLA", "F", "GM", "TM", "RIVN","TATAMOTORS.NS"]
}
tickers = []

for stocks in stock_dictionary.values():
    tickers.extend(stocks)

try:
    portfolio_data = yf.download(tickers, period="6mo")

except Exception as e:
    print("Download failed")
    print(e)

[                       0%                       ]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
[**********************52%                       ]  11 of 21 completed$TATAMOTORS.NS: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
[*********************100%***********************]  21 of 21 completed

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")


In [3]:
portfolio_data.shape

(123, 106)

In [4]:
portfolio_data.head(5)

Price          Adj Close       Close                                     \
Ticker     TATAMOTORS.NS        AAPL        ABBV        AMZN        BAC   
Date                                                                      
2025-12-23           NaN  271.854919  225.096466  232.139999  55.367695   
2025-12-24           NaN  273.302216  226.178711  232.380005  55.644680   
2025-12-26           NaN  272.893005  226.267258  232.520004  55.565536   
2025-12-29           NaN  273.252350  227.113373  232.070007  54.754364   
2025-12-30           NaN  272.573547  226.031143  232.529999  54.685120   

Price                                                                 ...  \
Ticker               C          F         GM        GOOG          GS  ...   
Date                                                                  ...   
2025-12-23  118.221634  12.983944  82.375969  315.268005  893.053589  ...   
2025-12-24  120.360313  13.052331  82.505379  315.258026  902.036438  ...   
2025-12-26  119.231567  13.003484  82.684563  314.548950  898.332275  ...   
2025-12-29  116.964165  12.974174  82.555153  313.979736  883.614990  ...   
2025-12-30  116.053246  12.925325  81.957863  314.139496  875.929443  ...   

Price         Volume                                                    \
Ticker           MRK       MS      MSFT       NVDA       PFE      RIVN   
Date                                                                     
2025-12-23  13587500  3514800  14683600  174873600  43692900  38763700   
2025-12-24   5335000  2648000   5855900   65528500  19328500  10629600   
2025-12-26   6280700  2564700   8842200  139740300  21617800  20503000   
2025-12-29   8086900  2464300  10893400  120006100  33067700  20707600   
2025-12-30   6497900  2365500  13944500   97687300  28821700  38607700   

Price                                                
Ticker     TATAMOTORS.NS      TM      TSLA      UNH  
Date                                                 
2025-12-23           NaN  330500  58223600  4463300  
2025-12-24           NaN  170800  41285400  2842700  
2025-12-26           NaN  146000  58780700  4359300  
2025-12-29           NaN  247100  66263000  4346800  
2025-12-30           NaN  184500  59238500  4432500  

[5 rows x 106 columns]

In [5]:
# data quality checks
failed_tickers = portfolio_data["Close"].isna().all()
failed_tickers = failed_tickers[failed_tickers]
failed_ticker_list = failed_tickers.index.tolist()
print("Failed tickers:")
print(failed_ticker_list)

Failed tickers:
['TATAMOTORS.NS']


In [6]:
portfolio_data = portfolio_data.drop(
    columns=failed_ticker_list,
    level=1
)

#### Clean dataframe presentation

In [7]:
latest_date = portfolio_data.index[-1]

In [8]:
clean_table = pd.DataFrame({
    "Close": portfolio_data["Close"].loc[latest_date],
    "High": portfolio_data["High"].loc[latest_date],
    "Low": portfolio_data["Low"].loc[latest_date],
    "Open": portfolio_data["Open"].loc[latest_date],
    "Volume": portfolio_data["Volume"].loc[latest_date]
})

In [9]:
clean_table.head()

,Close,High,Low,Open,Volume
Ticker,,,,,
AAPL,297.010010,302.420013,296.760010,297.309998,44812800
ABBV,230.009995,232.320007,221.940002,221.940002,9911500
AMZN,232.789993,242.000000,232.240005,240.080002,68233300
BAC,57.369999,57.730000,56.810001,56.849998,38651500
C,145.669998,146.369995,144.479996,144.710007,14976600


In [10]:
# need date column as well
clean_table = clean_table.reset_index()

In [11]:
clean_table.head(1)

,Ticker,Close,High,Low,Open,Volume
0,AAPL,297.01001,302.420013,296.76001,297.309998,44812800


In [12]:
# add date column
clean_table["Date"] = latest_date

In [13]:
# stock data of lastest date
clean_portfolio_data = clean_table[["Ticker","Date","Close","High","Low","Open","Volume"]]
clean_portfolio_data.head()

,Ticker,Date,Close,High,Low,Open,Volume
0,AAPL,2026-06-22,297.010010,302.420013,296.760010,297.309998,44812800
1,ABBV,2026-06-22,230.009995,232.320007,221.940002,221.940002,9911500
2,AMZN,2026-06-22,232.789993,242.000000,232.240005,240.080002,68233300
3,BAC,2026-06-22,57.369999,57.730000,56.810001,56.849998,38651500
4,C,2026-06-22,145.669998,146.369995,144.479996,144.710007,14976600


In [14]:
# map sectors of respective stocks 
ticker_sector = {}

for sector, stocks in stock_dictionary.items():
    for stock in stocks:
        ticker_sector[stock] = sector

In [15]:
clean_portfolio_data["Sector"] = clean_portfolio_data["Ticker"].map(ticker_sector)

C:\Users\Ashlesha\AppData\Local\Temp\ipykernel_4260\1232821456.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  clean_portfolio_data["Sector"] = clean_portfolio_data["Ticker"].map(ticker_sector)


In [16]:
clean_portfolio_data = clean_portfolio_data[["Ticker", "Sector", "Date", "Close", "High","Low", "Open", "Volume"]]

clean_portfolio_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume
0,AAPL,Technology,2026-06-22,297.010010,302.420013,296.760010,297.309998,44812800
1,ABBV,Healthcare,2026-06-22,230.009995,232.320007,221.940002,221.940002,9911500
2,AMZN,Technology,2026-06-22,232.789993,242.000000,232.240005,240.080002,68233300
3,BAC,Finance,2026-06-22,57.369999,57.730000,56.810001,56.849998,38651500
4,C,Finance,2026-06-22,145.669998,146.369995,144.479996,144.710007,14976600


In [17]:
clean_portfolio_data.isnull().sum()

Ticker    0
Sector    0
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [18]:
clean_portfolio_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Ticker  20 non-null     object        
 1   Sector  20 non-null     object        
 2   Date    20 non-null     datetime64[ns]
 3   Close   20 non-null     float64       
 4   High    20 non-null     float64       
 5   Low     20 non-null     float64       
 6   Open    20 non-null     float64       
 7   Volume  20 non-null     int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(2)
memory usage: 1.4+ KB


In [19]:
clean_portfolio_data['Date'] = pd.to_datetime(clean_portfolio_data['Date'],format='%d-%m-%Y')

In [20]:
clean_portfolio_data.describe()

,Date,Close,High,Low,Open,Volume
count,20,20.000000,20.000000,20.000000,20.000000,2.000000e+01
mean,2026-06-22 00:00:00,250.775498,254.737999,247.606252,250.919750,3.066776e+07
min,2026-06-22 00:00:00,14.110000,14.560000,13.950000,14.100000,7.421000e+05
25%,2026-06-22 00:00:00,106.717503,107.552498,104.767500,105.677502,9.563025e+06
50%,2026-06-22 00:00:00,228.549995,229.839996,223.465004,224.325005,2.110285e+07
75%,2026-06-22 00:00:00,335.805008,339.127495,330.297501,336.558762,4.487132e+07
max,2026-06-22 00:00:00,1106.369995,1115.979980,1091.630005,1109.699951,1.216787e+08
std,NaN,238.448644,240.969125,235.126122,238.968905,3.005868e+07


In [21]:
# check duplicates
clean_portfolio_data.duplicated().sum()

0

In [22]:
# clean dataset for the latest date
portfolio_data_with_sectors_lastestDay = clean_portfolio_data.copy()

In [23]:
portfolio_data.head(2)

Price            Close                                                 \
Ticker            AAPL        ABBV        AMZN        BAC           C   
Date                                                                    
2025-12-23  271.854919  225.096466  232.139999  55.367695  118.221634   
2025-12-24  273.302216  226.178711  232.380005  55.644680  120.360313   

Price                                                                 ...  \
Ticker              F         GM        GOOG          GS         JNJ  ...   
Date                                                                  ...   
2025-12-23  12.983944  82.375969  315.268005  893.053589  203.521378  ...   
2025-12-24  13.052331  82.505379  315.258026  902.036438  205.499420  ...   

Price        Volume                                                    \
Ticker          JPM       MRK       MS      MSFT       NVDA       PFE   
Date                                                                    
2025-12-23  6668300  13587500  3514800  14683600  174873600  43692900   
2025-12-24  4289300   5335000  2648000   5855900   65528500  19328500   

Price                                            
Ticker          RIVN      TM      TSLA      UNH  
Date                                             
2025-12-23  38763700  330500  58223600  4463300  
2025-12-24  10629600  170800  41285400  2842700  

[2 rows x 100 columns]

In [24]:
historical_stock_data = portfolio_data.copy()

In [25]:
historical_stock_data = (historical_stock_data.stack(level=1).reset_index())

historical_stock_data.head()

Price,Date,Ticker,Close,High,Low,Open,Volume
0,2025-12-23,AAPL,271.854919,271.994674,269.060124,270.337749,29642000
1,2025-12-23,ABBV,225.096466,227.064179,224.230683,224.535676,3612100
2,2025-12-23,AMZN,232.139999,232.449997,228.729996,229.059998,29230200
3,2025-12-23,BAC,55.367695,55.615005,55.209417,55.209417,22274600
4,2025-12-23,C,118.221634,119.083043,116.904758,116.993874,16135000


In [26]:
historical_stock_data.columns

Index(['Date', 'Ticker', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

In [27]:
# map sectors of respective stocks 
ticker_sector = {}

for sector, stocks in stock_dictionary.items():
    for stock in stocks:
        ticker_sector[stock] = sector

In [28]:
historical_stock_data["Sector"] = historical_stock_data["Ticker"].map(ticker_sector)

In [29]:
historical_stock_data = historical_stock_data[["Ticker", "Sector", "Date", "Close", "High","Low", "Open", "Volume"]]

In [30]:
historical_stock_data.head(2)

Price,Ticker,Sector,Date,Close,High,Low,Open,Volume
0,AAPL,Technology,2025-12-23,271.854919,271.994674,269.060124,270.337749,29642000
1,ABBV,Healthcare,2025-12-23,225.096466,227.064179,224.230683,224.535676,3612100


In [31]:
historical_stock_data.isnull().sum()

Price
Ticker    0
Sector    0
Date      0
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64

In [32]:
historical_stock_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2460 entries, 0 to 2459
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Ticker  2460 non-null   object        
 1   Sector  2460 non-null   object        
 2   Date    2460 non-null   datetime64[ns]
 3   Close   2460 non-null   float64       
 4   High    2460 non-null   float64       
 5   Low     2460 non-null   float64       
 6   Open    2460 non-null   float64       
 7   Volume  2460 non-null   int64         
dtypes: datetime64[ns](1), float64(4), int64(1), object(2)
memory usage: 153.9+ KB


In [33]:
historical_stock_data['Date'] = pd.to_datetime(historical_stock_data['Date'],format='%d-%m-%Y')

In [34]:
historical_stock_data.describe()

Price,Date,Close,High,Low,Open,Volume
count,2460,2460.000000,2460.000000,2460.000000,2460.000000,2.460000e+03
mean,2026-03-23 17:57:04.390244096,233.740092,236.586221,230.759673,233.634584,3.109291e+07
min,2025-12-23 00:00:00,11.070457,11.307469,10.971701,11.218589,1.460000e+05
25%,2026-02-06 00:00:00,99.139835,99.707145,97.380884,98.737747,7.187975e+06
50%,2026-03-24 00:00:00,212.518181,214.684998,209.534996,211.500000,1.653985e+07
75%,2026-05-07 00:00:00,305.579803,308.696558,301.596564,305.087502,4.152518e+07
max,2026-06-22 00:00:00,1106.369995,1125.000000,1094.290039,1120.000000,3.608079e+08
std,NaN,200.416334,203.038582,197.573866,200.236871,3.966408e+07


In [35]:
historical_stock_data.duplicated().sum()

0

In [36]:
# cleanups
historical_stock_data.columns.name = None
historical_stock_data = historical_stock_data.sort_values(["Ticker", "Date"])

In [37]:
historical_stock_data["Daily Return %"] = round(historical_stock_data.groupby('Ticker')['Close'].pct_change()*100,2)

In [38]:
historical_stock_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-23,271.854919,271.994674,269.060124,270.337749,29642000,NaN
20,AAPL,Technology,2025-12-24,273.302216,274.919206,271.695216,271.834940,17910600,0.53


In [39]:
historical_stock_data["Daily Return %"].describe()

count    2440.000000
mean        0.046061
std         2.140651
min       -19.610000
25%        -1.070000
50%        -0.000000
75%         1.132500
max        26.640000
Name: Daily Return %, dtype: float64

In [40]:
cumulative_returns = historical_stock_data.groupby('Ticker').agg(
                                                                 First_close = ('Close','first'),
                                                                 Last_close = ('Close','last'))
cumulative_returns

,First_close,Last_close
Ticker,,
AAPL,271.854919,297.010010
ABBV,225.096466,230.009995
AMZN,232.139999,232.789993
BAC,55.367695,57.369999
C,118.221634,145.669998
F,12.983944,14.110000
GM,82.375969,80.430000
GOOG,315.268005,348.779999
GS,893.053589,1106.369995


In [41]:
cumulative_returns["Cumulative Return %"] = round((cumulative_returns['Last_close'] - cumulative_returns['First_close']) / cumulative_returns['First_close'] *100 , 2)

cumulative_returns = cumulative_returns.sort_values("Cumulative Return %",ascending=False)
cumulative_returns.head()

,First_close,Last_close,Cumulative Return %
Ticker,,,
MS,177.561691,227.089996,27.89
UNH,320.464050,406.679993,26.90
GS,893.053589,1106.369995,23.89
C,118.221634,145.669998,23.22
JNJ,203.521378,231.289993,13.64


In [42]:
# average trading volume
cumulative_returns["Total Volume"] = (historical_stock_data.groupby("Ticker")["Volume"].sum())

# sector mapping
sector_map = (historical_stock_data.groupby("Ticker")["Sector"].first())
cumulative_returns["Sector"] = sector_map

In [43]:
stock_performance_summary = (cumulative_returns.reset_index())

stock_performance_summary = stock_performance_summary[["Ticker","Sector","First_close","Last_close","Cumulative Return %","Total Volume"]]
stock_performance_summary.head()

,Ticker,Sector,First_close,Last_close,Cumulative Return %,Total Volume
0,MS,Finance,177.561691,227.089996,27.89,809466900
1,UNH,Healthcare,320.464050,406.679993,26.90,1057251300
2,GS,Finance,893.053589,1106.369995,23.89,279109000
3,C,Finance,118.221634,145.669998,23.22,1667174100
4,JNJ,Healthcare,203.521378,231.289993,13.64,1006867500


In [44]:
# investor's porfolios 

investor_profiles = { 
    "Investor_A": {"capital": 10000, "weights": {"JPM": 0.30, "BAC": 0.30, "ABBV": 0.20, "JNJ": 0.20}},
    "Investor_B": { "capital": 10000,"weights": {"NVDA": 0.30, "TSLA": 0.25, "AMZN": 0.25, "AAPL": 0.20}}
}
investor_profiles

{'Investor_A': {'capital': 10000,
  'weights': {'JPM': 0.3, 'BAC': 0.3, 'ABBV': 0.2, 'JNJ': 0.2}},
 'Investor_B': {'capital': 10000,
  'weights': {'NVDA': 0.3, 'TSLA': 0.25, 'AMZN': 0.25, 'AAPL': 0.2}}}

In [45]:
for investor, details in investor_profiles.items():

    total_weight = sum(details["weights"].values())

    if total_weight != 1:
        raise ValueError(
            f"{investor} weights do not sum to 1. Current total: {total_weight}"
        )

-  validated that portfolio allocations summed to 100%, if not, the pipeline stops and raises an error to prevent incorrect return calculations

#### now lets calculate weighted return calculation for each investor

##### Investor A portfolio summary

In [46]:
investor_A_stocks = investor_profiles['Investor_A']['weights'].keys()
investor_A_stocks

dict_keys(['JPM', 'BAC', 'ABBV', 'JNJ'])

In [47]:
# filter the complete stock historical data to just gets stocks data held by the investor
investor_A_data = historical_stock_data[historical_stock_data['Ticker'].isin(investor_A_stocks)].copy()
investor_A_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
1,ABBV,Healthcare,2025-12-23,225.096466,227.064179,224.230683,224.535676,3612100,NaN
21,ABBV,Healthcare,2025-12-24,226.178711,227.074024,225.283398,225.755660,1744900,0.48
41,ABBV,Healthcare,2025-12-26,226.267258,226.887092,224.988242,226.109852,1593900,0.04
61,ABBV,Healthcare,2025-12-29,227.113373,228.274331,226.031140,226.434510,3803000,0.37
81,ABBV,Healthcare,2025-12-30,226.031143,227.477413,224.929214,227.044514,2348000,-0.48


In [48]:
investor_A_data['Weight'] = investor_A_data['Ticker'].map(investor_profiles['Investor_A']['weights'])

In [49]:
investor_A_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %,Weight
1,ABBV,Healthcare,2025-12-23,225.096466,227.064179,224.230683,224.535676,3612100,NaN,0.2
21,ABBV,Healthcare,2025-12-24,226.178711,227.074024,225.283398,225.755660,1744900,0.48,0.2


In [50]:
investor_A_data["Weighted Return"] = (investor_A_data["Daily Return %"] * investor_A_data["Weight"])

In [51]:
portfolio_A_daily_returns = (investor_A_data.groupby("Date")["Weighted Return"].sum().reset_index())
portfolio_A_daily_returns.rename(columns={"Weighted Return": "Portfolio Daily Return %"},inplace = True )

In [52]:
portfolio_A_daily_returns.head()

,Date,Portfolio Daily Return %
0,2025-12-23,0.000
1,2025-12-24,0.737
2,2025-12-26,-0.162
3,2025-12-29,-0.751
4,2025-12-30,-0.227


In [53]:
initial_capital = investor_profiles["Investor_A"]["capital"]

portfolio_A_daily_returns["Portfolio Value"] = round((initial_capital * 
                                                (1 + portfolio_A_daily_returns["Portfolio Daily Return %"] / 100).cumprod()),3)
portfolio_A_daily_returns.head()

,Date,Portfolio Daily Return %,Portfolio Value
0,2025-12-23,0.000,10000.000
1,2025-12-24,0.737,10073.700
2,2025-12-26,-0.162,10057.381
3,2025-12-29,-0.751,9981.850
4,2025-12-30,-0.227,9959.191


In [54]:
portfolio_A_daily_returns["Portfolio Cumulative Return %"] = round(((portfolio_A_daily_returns["Portfolio Value"] - initial_capital) / initial_capital )*100,3)
portfolio_A_daily_returns.head(3)

,Date,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-23,0.000,10000.000,0.000
1,2025-12-24,0.737,10073.700,0.737
2,2025-12-26,-0.162,10057.381,0.574


##### Investor B portfolio summary

In [55]:
investor_B_stocks = investor_profiles['Investor_B']['weights'].keys()
investor_B_stocks

dict_keys(['NVDA', 'TSLA', 'AMZN', 'AAPL'])

In [56]:
# filter the complete stock historical data to just gets stocks data held by the investor
investor_B_data = historical_stock_data[historical_stock_data['Ticker'].isin(investor_B_stocks)].copy()
investor_B_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-23,271.854919,271.994674,269.060124,270.337749,29642000,NaN
20,AAPL,Technology,2025-12-24,273.302216,274.919206,271.695216,271.834940,17910600,0.53
40,AAPL,Technology,2025-12-26,272.893005,274.859353,272.353998,273.651606,21521800,-0.15
60,AAPL,Technology,2025-12-29,273.252350,273.851213,271.844961,272.184327,23715200,0.13
80,AAPL,Technology,2025-12-30,272.573547,273.571693,271.775043,272.304059,22139600,-0.25


In [57]:
investor_B_data['Weight'] = investor_B_data['Ticker'].map(investor_profiles['Investor_B']['weights'])

In [58]:
investor_B_data.head(2)

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %,Weight
0,AAPL,Technology,2025-12-23,271.854919,271.994674,269.060124,270.337749,29642000,NaN,0.2
20,AAPL,Technology,2025-12-24,273.302216,274.919206,271.695216,271.834940,17910600,0.53,0.2


In [59]:
investor_B_data["Weighted Return"] = (investor_B_data["Daily Return %"] * investor_B_data["Weight"])

In [60]:
portfolio_B_daily_returns = (investor_B_data.groupby("Date")["Weighted Return"].sum().reset_index())
portfolio_B_daily_returns.rename(columns={"Weighted Return": "Portfolio Daily Return %"},inplace = True )

In [61]:
portfolio_B_daily_returns.head()

,Date,Portfolio Daily Return %
0,2025-12-23,0.0000
1,2025-12-24,0.0275
2,2025-12-26,-0.2340
3,2025-12-29,-1.2020
4,2025-12-30,-0.3905


In [62]:
initial_capital = investor_profiles["Investor_B"]["capital"]

portfolio_B_daily_returns["Portfolio Value"] = round((initial_capital * 
                                                (1 + portfolio_B_daily_returns["Portfolio Daily Return %"] / 100).cumprod()),3)
portfolio_B_daily_returns.head()

,Date,Portfolio Daily Return %,Portfolio Value
0,2025-12-23,0.0000,10000.000
1,2025-12-24,0.0275,10002.750
2,2025-12-26,-0.2340,9979.344
3,2025-12-29,-1.2020,9859.392
4,2025-12-30,-0.3905,9820.891


In [63]:
portfolio_B_daily_returns["Portfolio Cumulative Return %"] = round(((portfolio_B_daily_returns["Portfolio Value"] - initial_capital) / initial_capital )*100,3)
portfolio_B_daily_returns.head(3)

,Date,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-23,0.0000,10000.000,0.000
1,2025-12-24,0.0275,10002.750,0.028
2,2025-12-26,-0.2340,9979.344,-0.207


In [64]:
portfolio_A_daily_returns.iloc[-1]

Date                             2026-06-22 00:00:00
Portfolio Daily Return %                       2.704
Portfolio Value                            10579.917
Portfolio Cumulative Return %                  5.799
Name: 122, dtype: object

In [65]:
portfolio_B_daily_returns.iloc[-1]

Date                             2026-06-22 00:00:00
Portfolio Daily Return %                     -1.2615
Portfolio Value                            10166.511
Portfolio Cumulative Return %                  1.665
Name: 122, dtype: object

In [66]:
return_A = portfolio_A_daily_returns["Portfolio Cumulative Return %"].iloc[-1]
return_B = portfolio_B_daily_returns["Portfolio Cumulative Return %"].iloc[-1]

In [67]:
# expected annual return is

annual_return_A = round((portfolio_A_daily_returns["Portfolio Daily Return %"].mean()* 252),2)
annual_return_B = round((portfolio_B_daily_returns["Portfolio Daily Return %"].mean()* 252),2)
print(annual_return_A)
print(annual_return_B)

12.76
6.34


#### calculate risk to check which portfolio is more volatile

In [68]:
portfolio_A_daily_returns["Portfolio Daily Return %"].std()

0.9851380303411651

In [69]:
portfolio_B_daily_returns["Portfolio Daily Return %"].std()

1.53718707686788

In [70]:
annual_risk_A = (portfolio_A_daily_returns["Portfolio Daily Return %"].std() * (252 ** 0.5))

annual_risk_B = (portfolio_B_daily_returns["Portfolio Daily Return %"].std() * (252 ** 0.5))

print(round(annual_risk_A,2))
print(round(annual_risk_B,2))

15.64
24.4


In [71]:
# assuming risk-free rate as 0 
sharpe_A = round(annual_return_A / annual_risk_A,2)
sharpe_B = round(annual_return_B / annual_risk_B,2)
print(sharpe_A)
print(sharpe_B)

0.82
0.26


In [72]:
# creating the overall summary
portfolio_summary = pd.DataFrame({
                                "Portfolio": ["Conservative", "Growth"],
                                "Capital": [10000, 10000],
                                "Current Value": [
                                        round(portfolio_A_daily_returns["Portfolio Value"].iloc[-1], 2),
                                        round(portfolio_B_daily_returns["Portfolio Value"].iloc[-1], 2)],
                                "Total Return %": [
                                        round(portfolio_A_daily_returns["Portfolio Cumulative Return %"].iloc[-1], 2),
                                        round(portfolio_B_daily_returns["Portfolio Cumulative Return %"].iloc[-1], 2)],
                                "Annual Return %": [
                                        annual_return_A,
                                        annual_return_B],
                                "Annual Risk %": [
                                        round(annual_risk_A, 2),
                                        round(annual_risk_B, 2)],
                                "Sharpe Ratio": [
                                        sharpe_A,
                                        sharpe_B]
                                })

In [73]:
portfolio_summary.head()

,Portfolio,Capital,Current Value,Total Return %,Annual Return %,Annual Risk %,Sharpe Ratio
0,Conservative,10000,10579.92,5.80,12.76,15.64,0.82
1,Growth,10000,10166.51,1.66,6.34,24.40,0.26


In [74]:
portfolio_A_daily_returns["Portfolio"] = "Conservative"

In [75]:
portfolio_B_daily_returns["Portfolio"] = "Growth"

In [76]:
portfolio_daily_returns = pd.concat([portfolio_A_daily_returns, portfolio_B_daily_returns],ignore_index=True)

In [77]:
portfolio_daily_returns = portfolio_daily_returns[["Date","Portfolio","Portfolio Daily Return %","Portfolio Value",
                                                   "Portfolio Cumulative Return %"]]

In [78]:
portfolio_daily_returns = portfolio_daily_returns.sort_values(["Date", "Portfolio"])

In [79]:
portfolio_daily_returns = portfolio_daily_returns.reset_index(drop=True)

In [80]:
portfolio_daily_returns.head()

,Date,Portfolio,Portfolio Daily Return %,Portfolio Value,Portfolio Cumulative Return %
0,2025-12-23,Conservative,0.0000,10000.000,0.000
1,2025-12-23,Growth,0.0000,10000.000,0.000
2,2025-12-24,Conservative,0.7370,10073.700,0.737
3,2025-12-24,Growth,0.0275,10002.750,0.028
4,2025-12-26,Conservative,-0.1620,10057.381,0.574


In [81]:
historical_stock_data = historical_stock_data.sort_values(["Ticker", "Date"])

In [82]:
historical_stock_data.head()

,Ticker,Sector,Date,Close,High,Low,Open,Volume,Daily Return %
0,AAPL,Technology,2025-12-23,271.854919,271.994674,269.060124,270.337749,29642000,NaN
20,AAPL,Technology,2025-12-24,273.302216,274.919206,271.695216,271.834940,17910600,0.53
40,AAPL,Technology,2025-12-26,272.893005,274.859353,272.353998,273.651606,21521800,-0.15
60,AAPL,Technology,2025-12-29,273.252350,273.851213,271.844961,272.184327,23715200,0.13
80,AAPL,Technology,2025-12-30,272.573547,273.571693,271.775043,272.304059,22139600,-0.25


In [83]:
print(historical_stock_data.shape)
print(portfolio_daily_returns.shape)
print(portfolio_summary.shape)

(2460, 9)
(246, 5)
(2, 7)


In [84]:
historical_stock_data = historical_stock_data.reset_index(drop=True)
portfolio_daily_returns = portfolio_daily_returns.reset_index(drop=True)
portfolio_summary = portfolio_summary.reset_index(drop=True)

In [85]:
historical_stock_data.to_csv("historical_stock_data.csv", index=False)
portfolio_daily_returns.to_csv("portfolio_daily_returns.csv", index=False)
portfolio_summary.to_csv("portfolio_summary.csv", index=False)
stock_performance_summary.to_csv("stock_performance_summary.csv",index=False)

### generate PDF report

In [86]:
import pandas as pd

In [87]:
portfolio_summary = pd.read_csv("portfolio_summary.csv")

portfolio_summary

,Portfolio,Capital,Current Value,Total Return %,Annual Return %,Annual Risk %,Sharpe Ratio
0,Conservative,10000,10579.92,5.80,12.76,15.64,0.82
1,Growth,10000,10166.51,1.66,6.34,24.40,0.26


In [88]:
max_return  = portfolio_summary["Total Return %"].max()
max_return

5.8

In [89]:
best_portfolio = portfolio_summary.loc[portfolio_summary["Total Return %"]==max_return,"Portfolio"].iloc[0]
worst_portfolio = portfolio_summary.loc[portfolio_summary["Total Return %"] == portfolio_summary["Total Return %"].min(),"Portfolio"].iloc[0]

In [90]:
print(best_portfolio)
print(worst_portfolio)

Conservative
Growth


In [91]:
return_difference = round(portfolio_summary["Total Return %"].max() - portfolio_summary["Total Return %"].min(),2) 

In [92]:
return_difference

4.14

In [93]:
print(f"Best Performing Portfolio: {best_portfolio}")
print(f"{best_portfolio} Portfolio Outperformed {worst_portfolio} Portfolio by {return_difference}%")

Best Performing Portfolio: Conservative
Conservative Portfolio Outperformed Growth Portfolio by 4.14%


In [94]:
portfolio_daily_returns = pd.read_csv("portfolio_daily_returns.csv")

In [95]:
report_date = portfolio_daily_returns["Date"].max()
report_date

'2026-06-22'

In [96]:
report_text = f"""
📊 Portfolio Report

Report Date: {report_date}

"""

In [97]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-22




In [98]:
for _, portfolio in portfolio_summary.iterrows():

    report_text += f"""

Portfolio: {portfolio["Portfolio"]}

Current Value: ₹{portfolio["Current Value"]}

Total Return: {portfolio["Total Return %"]}%

Annual Return: {portfolio["Annual Return %"]}%

Annual Risk: {portfolio["Annual Risk %"]}%

Sharpe Ratio: {portfolio["Sharpe Ratio"]}

"""

In [99]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-22



Portfolio: Conservative

Current Value: ₹10579.92

Total Return: 5.8%

Annual Return: 12.76%

Annual Risk: 15.64%

Sharpe Ratio: 0.82



Portfolio: Growth

Current Value: ₹10166.51

Total Return: 1.66%

Annual Return: 6.34%

Annual Risk: 24.4%

Sharpe Ratio: 0.26




In [100]:
report_text += f"""

Portfolio Comparison

Best Performing Portfolio: {best_portfolio}

{best_portfolio} Portfolio Outperformed Conservative Portfolio by {return_difference}%

"""

In [101]:
print(report_text)


📊 Portfolio Report

Report Date: 2026-06-22



Portfolio: Conservative

Current Value: ₹10579.92

Total Return: 5.8%

Annual Return: 12.76%

Annual Risk: 15.64%

Sharpe Ratio: 0.82



Portfolio: Growth

Current Value: ₹10166.51

Total Return: 1.66%

Annual Return: 6.34%

Annual Risk: 24.4%

Sharpe Ratio: 0.26



Portfolio Comparison

Best Performing Portfolio: Conservative

Conservative Portfolio Outperformed Conservative Portfolio by 4.14%




!pip install reportlab

In [102]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

In [103]:
# creating a variable to save the path for report on my system
pdf_path = f"reports/portfolio_report_{report_date}.pdf"

In [104]:
# create an empty pdf document object 
doc = SimpleDocTemplate(pdf_path)

In [105]:
# get default text formatting styles
styles = getSampleStyleSheet()

In [106]:
elements = []

In [107]:
# add Title and recent date
elements.append(Paragraph("Portfolio Performance Report", styles["Title"]))

elements.append(Paragraph(f"Report Date: {report_date}", styles["Normal"]))

elements.append(Spacer(1, 12))

In [108]:
# add portfolio details of each
for _, portfolio in portfolio_summary.iterrows():

    elements.append(Paragraph(f"<b>Portfolio:</b> {portfolio['Portfolio']}",styles["Heading2"]))

    elements.append(Paragraph(f"Current Value: ₹{portfolio['Current Value']}",styles["Normal"]))

    elements.append(Paragraph(f"Total Return: {portfolio['Total Return %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Annual Return: {portfolio['Annual Return %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Annual Risk: {portfolio['Annual Risk %']}%",styles["Normal"]))

    elements.append(Paragraph(f"Sharpe Ratio: {portfolio['Sharpe Ratio']}",styles["Normal"]))

    elements.append(Spacer(1, 12))

In [109]:
# add the comparision section
elements.append(Paragraph("Portfolio Comparison", styles["Heading1"]))

elements.append(Paragraph(f"Best Performing Portfolio: {best_portfolio}",styles["Normal"]))

elements.append(Paragraph(f"{best_portfolio} Portfolio Outperformed Conservative Portfolio by {return_difference}%",styles["Normal"]))

In [110]:
print(len(elements))

20


In [111]:
doc.build(elements)

print("PDF Report Created Successfully!")

PDF Report Created Successfully!
